# 12.3: Arena-style ratings: Elo and Bradley-Terry

where $\sigma$ is the logistic sigmoid. Fitting the $\beta$'s by maximum likelihood is just logistic regression: each vote is a training example whose features are $+1$ for model $i$, $-1$ for model $j$, and 0 elsewhere. The Chatbot Arena paper describes this approach, reports scores on an Elo-like scale for familiarity, and computes confidence intervals by bootstrap resampling of the votes. The Arena has also added "style control," which adds features such as response length and markdown formatting to the regression so that rankings reflect content more than presentation, the same idea as length-controlled AlpacaEval.

A small Bradley-Terry fit takes a few lines:


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

models = ["A", "B", "C"]
# Each vote: (winner, loser)
votes = [("A", "B")] * 60 + [("B", "A")] * 40 + [("B", "C")] * 55 + \
        [("C", "B")] * 45 + [("A", "C")] * 70 + [("C", "A")] * 30

idx = {m: i for i, m in enumerate(models)}
X, y = [], []
for w, l in votes:
    row = np.zeros(len(models)); row[idx[w]], row[idx[l]] = 1, -1
    X.append(row); y.append(1)
    X.append(-row); y.append(0)          # mirrored copy keeps the problem symmetric
clf = LogisticRegression(fit_intercept=False, C=1e6).fit(np.array(X), np.array(y))
beta = clf.coef_[0]
elo_like = 1000 + 400 / np.log(10) * (beta - beta.mean())   # rescale to an Elo-style scale
print(dict(zip(models, elo_like.round())))
